In [ ]:
# from spd.models.component_utils import (
#     calc_component_acts,
#     calc_mask_l_zero,
#     calc_masks,
#     calc_random_masks,
#     component_activation_statistics,
# )
# from spd.models.components import EmbeddingComponent, Gate, GateMLP, LinearComponent
# from transformers import AutoTokenizer
# import torch
# from spd.models.component_model import ComponentModel

# device = "cuda" if torch.cuda.is_available() else "cpu"
# # model_path = "out/all_kl_06-27_15.58_20250627_155830_844/model_50000.pth"
# model_path = "out/last_mlp/model_50000.pth"
# model, config, out_dir = ComponentModel.from_pretrained(model_path)
# model.to(device)
# model.requires_grad_(False)
# model_name = "roneneldan/TinyStories-1M"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# module_comp = [key.replace('-', '.') for key in model.components.keys()]


# text = " Once upon a time, a boy named"
# tokens = torch.tensor([tokenizer.encode(text), tokenizer.encode(" Then suddenly, a dragon appeared. So")])

# gates: dict[str, Gate | GateMLP] = {
#     k.removeprefix("gates.").replace("-", "."): v for k, v in model.gates.items()
# }  # type: ignore
# components: dict[str, LinearComponent | EmbeddingComponent] = {
#     k.removeprefix("components.").replace("-", "."): v for k, v in model.components.items()
# }  
# As = {module_name: components[module_name].A for module_name in components}
# with torch.no_grad():
#     orig_logits, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
#         tokens.to(device),
#         module_names = module_comp
#     )
#     target_component_acts = calc_component_acts(pre_weight_acts=pre_weight_acts, As=As)  # type: ignore


#     masks, sparsity_masks = calc_masks(
#         gates=gates, target_component_acts=target_component_acts, detach_inputs=False
#     )

#     # new_mask = masks.copy()
#     # new_mask['transformer.h.7.mlp.c_fc'][0, 6, 1534]  = 0.0

#     logits = model.forward_with_components(
#         tokens.to(device),
#         # components=model.components,
#         components=components,
#         masks=masks
#         # masks=new_mask,
#     )
# seq_pos = -2
# val, idx = logits[:, seq_pos].topk(10)
# val_o, idx_o = orig_logits[:, seq_pos].topk(10)
# tokenizer.batch_decode(idx), tokenizer.batch_decode(idx_o)

In [22]:
from spd.models.component_utils import (
    calc_component_acts,
    calc_mask_l_zero,
    calc_masks,
    calc_random_masks,
    component_activation_statistics,
)
from spd.models.components import EmbeddingComponent, Gate, GateMLP, LinearComponent
from transformers import AutoTokenizer
import torch
from spd.models.component_model import ComponentModel
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from collections import defaultdict

device = "cuda" if torch.cuda.is_available() else "cpu"
# model_path = "out/last_mlp/model_50000.pth"
# model_path = "out/decent_run/model_50000.pth"
model_path = "out/decent_run_100k/model_100000.pth"
model, config, out_dir = ComponentModel.from_pretrained(model_path)
model.to(device)
model.requires_grad_(False)
model_name = "roneneldan/TinyStories-1M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Create dataloader
dataset = load_dataset("roneneldan/TinyStories", split="train[:10_000]")


# Track specific components
target_modules = {
    "transformer.h.2.mlp.c_fc": [i for i in range(100)],  # module: component_idx
}
# # Track specific components
# target_modules = {
#     "transformer.h.7.mlp.c_fc": [i for i in range(20)],  # Track components 0-4
#     "transformer.h.7.mlp.c_proj": [i for i in range(20)]  # Track specific components
# }

def collate_fn(examples):
    texts = [ex['text'] for ex in examples]
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors='pt'
    )
    return tokenized

dataloader = DataLoader(
    dataset,
    batch_size=32,
    collate_fn=collate_fn,
    shuffle=False
)
def gather_component_activations_indexed(
    model: ComponentModel,
    tokenizer: AutoTokenizer,
    dataloader: DataLoader,
    device: str = "cuda",
    max_datapoints: int = 10000,
    target_modules: dict[str, list[int]] = None,  # {"module_name": [comp_idx1, comp_idx2, ...]}
) -> tuple[dict[str, torch.Tensor], torch.Tensor]:
    """
    Gather component activations with fully batched processing.
    
    Returns:
        - Dict mapping module_name -> tensor of shape [n_datapoints, seq_len, n_tracked_components]
        - token_sequences: Tensor of shape [n_datapoints, seq_len]
    """
    # Get module names and components
    module_comp = [key.replace('-', '.') for key in model.components.keys()]
    components = {
        k.removeprefix("components.").replace("-", "."): v for k, v in model.components.items()
    }
    As = {module_name: components[module_name].A for module_name in components}
    
    # If target_modules not specified, track all
    if target_modules is None:
        target_modules = {module_name: None for module_name in module_comp}
    
    # Pre-allocate storage
    seq_len = 64  # from your config
    
    # Storage for activations and tokens
    activations_storage = {}
    for module_name, comp_indices in target_modules.items():
        if comp_indices is not None:
            n_components = len(comp_indices)
        else:
            # Get number of components from model
            n_components = components[module_name].A.shape[0]
        activations_storage[module_name] = torch.zeros(max_datapoints, seq_len, n_components)
    
    token_sequences = torch.zeros(max_datapoints, seq_len, dtype=torch.long)
    
    total_processed = 0
    
    with torch.no_grad():
        for batch_idx, batch in tqdm(enumerate(dataloader)):
            # print(f"Batch {batch_idx}")
            if total_processed >= max_datapoints:
                break
                
            input_ids = batch['input_ids'].to(device)
            batch_size = input_ids.shape[0]
            
            # Determine how many to process
            n_to_process = min(batch_size, max_datapoints - total_processed)
            
            # Store token sequences
            token_sequences[total_processed:total_processed + n_to_process] = input_ids[:n_to_process].cpu()
            
            # Get activations
            _, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
                input_ids[:n_to_process],
                module_names=module_comp
            )
            
            # Calculate component activations
            target_component_acts = calc_component_acts(
                pre_weight_acts=pre_weight_acts, 
                As=As
            )
            
            # Process target modules
            for module_name, comp_indices in target_modules.items():
                if module_name not in target_component_acts:
                    continue
                
                # Get activations: [batch_size, seq_len, n_components]
                acts = target_component_acts[module_name]
                
                if comp_indices is not None:
                    # Index into specific components
                    acts = acts[:, :, comp_indices]
                
                # Store
                activations_storage[module_name][total_processed:total_processed + n_to_process] = acts.cpu()
            
            total_processed += n_to_process
            
            # Clear GPU memory
            del input_ids, pre_weight_acts, target_component_acts
            torch.cuda.empty_cache()
    
    # Trim to actual size processed
    token_sequences = token_sequences[:total_processed]
    for module_name in activations_storage:
        activations_storage[module_name] = activations_storage[module_name][:total_processed]
    
    return activations_storage, token_sequences

# Example usage:
print("Gathering component activations...")


activations_storage, token_sequences = gather_component_activations_indexed(
    model=model,
    tokenizer=tokenizer,
    dataloader=dataloader,
    device=device,
    max_datapoints=10_000,
    target_modules=target_modules
)

# Print shapes
print(f"\nToken sequences shape: {token_sequences.shape}")
for module_name, acts in activations_storage.items():
    print(f"{module_name} activations shape: {acts.shape}")


Gathering component activations...


313it [00:10, 28.60it/s]


Token sequences shape: torch.Size([10000, 64])
transformer.h.2.mlp.c_fc activations shape: torch.Size([10000, 64, 100])


In [ ]:
def visualize_top_component_activations(
    activations_storage: dict[str, torch.Tensor],
    token_sequences: torch.Tensor,
    tokenizer: AutoTokenizer,
    module_name: str,
    component_idx: int,
    top_k: int = 10,
    context_before: int = 10,
    context_after: int = 5,
) -> str:
    """
    Create HTML visualization for top K activations of a specific component.
    
    Args:
        activations_storage: Dict from gather_component_activations_indexed
        token_sequences: Token sequences tensor
        tokenizer: Tokenizer for decoding
        module_name: Which module to visualize
        component_idx: Which component index within that module
        top_k: Number of top activations to show
        context_before: Tokens to show before activation
        context_after: Tokens to show after activation
    
    Returns:
        HTML string
    """
    # Get activations for this component
    # Note: component_idx here is the index within the tracked components, not the original component index
    component_acts = activations_storage[module_name][:, :, component_idx]
    
    # Find top K activations
    flat_acts = component_acts.flatten()
    top_k_values, top_k_indices = torch.topk(flat_acts, k=min(top_k, flat_acts.numel()))
    
    # Convert flat indices back to (datapoint, position)
    n_datapoints, seq_len = component_acts.shape
    top_k_locations = []
    for idx in top_k_indices:
        datapoint_idx = idx // seq_len
        token_pos = idx % seq_len
        activation_value = component_acts[datapoint_idx, token_pos].item()
        top_k_locations.append((datapoint_idx.item(), token_pos.item(), activation_value))
    
    # Get activation range for this component
    nonzero_acts = component_acts[component_acts > 0]
    if len(nonzero_acts) > 0:
        min_act = nonzero_acts.min().item()
        max_act = nonzero_acts.max().item()
    else:
        min_act, max_act = 0, 1
    
    # Start building HTML
    html_parts = [
        '<div style="font-family: monospace; background-color: transparent; padding: 20px;">',
        f'<h2>Component {component_idx} - Module: {module_name}</h2>',
    ]
    
    # Add color bar
    html_parts.append('<div style="margin-bottom: 20px;">')
    html_parts.append('<span>Activation strength: </span>')
    html_parts.append(make_colorbar(min_act, max_act))
    html_parts.append('</div>')
    
    # Process each top activation
    for rank, (datapoint_idx, token_pos, act_value) in enumerate(top_k_locations):
        html_parts.append(f'<div style="margin-bottom: 15px; padding-left: 0;">')

        # html_parts.append(f'<div style="margin-bottom: 15px; border: 1px solid #ccc; padding: 10px;">')
        # html_parts.append(f'<div style="margin-bottom: 15px; border: 1px solid #ccc; padding: 10px;">')
        # html_parts.append(f'<div><strong>Rank {rank + 1}</strong> - Activation: {act_value:.3f}</div>')
        
        # Get tokens for this datapoint
        tokens = token_sequences[datapoint_idx]
        
        # Determine context window
        start_pos = max(0, token_pos - context_before)
        end_pos = min(len(tokens), token_pos + context_after + 1)
        
        # Get activations for this sequence
        seq_acts = component_acts[datapoint_idx]
        
        # Build token display
        token_html = []
        for pos in range(start_pos, end_pos):
            token = tokens[pos].item()
            if token == 0:  # Skip padding
                continue
                
            token_text = tokenizer.decode([token])
            activation = seq_acts[pos].item() if pos < len(seq_acts) else 0
            
            # Determine background color
            if activation > 0:
                # Scale activation to color intensity
                intensity = activation / max_act if max_act > 0 else 0
                # Blue color with varying intensity
                blue_value = 255
                red_green_value = int(255 * (1 - intensity * 0.7))  # Scale from white to blue
                bg_color = f"rgb({red_green_value}, {red_green_value}, {blue_value})"
                text_color = "black" if intensity < 0.5 else "white"
            else:
                bg_color = "white"
                text_color = "black"
            
            # Special formatting for the target position
            if pos == token_pos:
                token_html.append(
                    f'<span style="background-color: {bg_color}; color: {text_color}; '
                    f'border: 2px solid red; padding: 2px;">{token_text}</span>'
                )
            else:
                token_html.append(
                    f'<span style="background-color: {bg_color}; color: {text_color}; '
                    f'padding: 2px;">{token_text}</span>'
                )
        
        html_parts.append('<div style="margin-top: 5px;">')
        html_parts.append(''.join(token_html))
        html_parts.append('</div>')
        html_parts.append('</div>')
    
    html_parts.append('</div>')
    return ''.join(html_parts)

def make_colorbar(min_value: float, max_value: float) -> str:
    """Create a color bar showing the activation range."""
    html = []
    n_steps = 5
    
    for i in range(n_steps + 1):
        ratio = i / n_steps
        value = min_value + (max_value - min_value) * ratio
        
        # Calculate color
        blue_value = 255
        red_green_value = int(255 * (1 - ratio * 0.7))
        bg_color = f"rgb({red_green_value}, {red_green_value}, {blue_value})"
        text_color = "black" if ratio < 0.5 else "white"
        
        html.append(
            f'<span style="background-color: {bg_color}; color: {text_color}; '
            f'padding: 4px 8px; margin-right: 2px;">{value:.3f}</span>'
        )
    
    return ''.join(html)

# Example usage:
from IPython.display import display, HTML

for i in range(30):
    # Visualize component 0 from the first tracked module
    module_name = list(activations_storage.keys())[0]
    # module_name = list(activations_storage.keys())[1]
    html = visualize_top_component_activations(
        activations_storage=activations_storage,
        token_sequences=token_sequences,
        tokenizer=tokenizer,
        module_name=module_name,
        component_idx=i,  # This is the index within the tracked components
        top_k=20,
        context_before=15,
        context_after=5
    )

    display(HTML(html))

In [ ]:
# Your handwritten text
text = "Once upon a time there was an ice-cream"
token_position = 8  # Specify which token position to analyze

# Tokenize
inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
input_ids = inputs['input_ids'].to(device)

# Print tokens to help identify positions
text_list = [str(tok_idx) + ": " + tokenizer.decode(token_ids) for tok_idx, token_ids in enumerate(input_ids[0])]
print(text_list)

# Verify the token at specified position
target_token_id = input_ids[0, token_position].item()
target_token_text = tokenizer.decode([target_token_id])
print(f"\n>>> Analyzing position {token_position}: '{target_token_text}' <<<\n")

# Get activations
with torch.no_grad():
    _, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
        input_ids,
        module_names=list(target_modules.keys())
    )

# Calculate component activations
As = {module_name: model.components[f"{module_name.replace('.', '-')}"].A 
      for module_name in target_modules.keys()}
component_acts = calc_component_acts(pre_weight_acts=pre_weight_acts, As=As)

# Find top activating components AT THE SPECIFIED POSITION
top_components = []
for module_name, acts in component_acts.items():
    # acts shape: [1, seq_len, n_components]
    # Get activations only at the specified position
    acts_at_position = acts[0, token_position, :]  # [n_components]
    
    # Get top 10 components for this module at this position
    top_values, top_indices = torch.topk(acts_at_position, k=min(10, len(acts_at_position)))
    
    for idx, val in zip(top_indices, top_values):
        if val > 0:  # Only include components that actually activated
            top_components.append((module_name, idx.item(), val.item()))

# Sort by activation value
top_components.sort(key=lambda x: x[2], reverse=True)

# Create target_modules dict from the top components found
target_modules_from_top = {}
for module_name, comp_idx, _ in top_components[:10]:  # Track top 10 components
    if module_name not in target_modules_from_top:
        target_modules_from_top[module_name] = []
    target_modules_from_top[module_name].append(comp_idx)

print(f"\nGathering activations for components: {target_modules_from_top}")

# Gather activations from larger dataset
activations_storage, token_sequences = gather_component_activations_indexed(
    model=model,
    tokenizer=tokenizer,
    dataloader=dataloader,
    device=device,
    max_datapoints=1000,
    target_modules=target_modules_from_top
)

# Visualize each top component
# print("\n\nVisualizations:")
for i, (module_name, comp_idx, activation_value) in enumerate(top_components[:10]):
    # print(f"\n=== Component {comp_idx} from {module_name} ===")
    # print(f"Activation on '{target_token_text}' at position {token_position}: {activation_value:.3f}")
    
    # First show the ice-cream example
    activations_storage_single = {
        module_name: component_acts[module_name][:, :, [comp_idx]].cpu()
    }
    token_sequences_single = input_ids.cpu()
    
    html_icecream = visualize_top_component_activations(
        activations_storage=activations_storage_single,
        token_sequences=token_sequences_single,
        tokenizer=tokenizer,
        module_name=module_name,
        component_idx=0,
        top_k=1,
        context_before=30,
        context_after=30
    )
    
    # # Modify title for ice-cream example
    html_icecream = html_icecream.replace(
        f'<h2>Component 0 -', 
        f'<h2>Your example - Component {comp_idx} -'
    )
    
    display(HTML(html_icecream))
    
    # Now show top activations from the larger dataset
    # Find the index of this component in our tracked components
    comp_index_in_tracked = target_modules_from_top[module_name].index(comp_idx)
    
    html_dataset = visualize_top_component_activations(
        activations_storage=activations_storage,
        token_sequences=token_sequences,
        tokenizer=tokenizer,
        module_name=module_name,
        component_idx=comp_index_in_tracked,
        top_k=10,  # Show top 10 from dataset
        context_before=15,
        context_after=5
    )
    
    # # Modify title for dataset examples
    html_dataset = html_dataset.replace(
        f'<h2>Component {comp_index_in_tracked} -', 
        f'<h2>Top activations from dataset - Component {comp_idx} -'
    )
    
    display(HTML(html_dataset))

In [ ]:
tokenizer.batch_decode(input_ids)

In [ ]:
# Your handwritten text
text = "Once upon a time there was an ice-cream"
token_position = 8  # Specify which token position to analyze

# Tokenize
inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
input_ids = inputs['input_ids'].to(device)

# Print tokens to help identify positions
text_list = [str(tok_idx) + ": " + tokenizer.decode(token_ids) for tok_idx, token_ids in enumerate(input_ids[0])]
print(text_list)

# Verify the token at specified position
target_token_id = input_ids[0, token_position].item()
target_token_text = tokenizer.decode([target_token_id])
print(f"\n>>> Analyzing position {token_position}: '{target_token_text}' <<<\n")

# Get activations
with torch.no_grad():
    _, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
        input_ids,
        module_names=list(target_modules.keys())
    )

# Calculate component activations
As = {module_name: model.components[f"{module_name.replace('.', '-')}"].A 
      for module_name in target_modules.keys()}
component_acts = calc_component_acts(pre_weight_acts=pre_weight_acts, As=As)

# Find top activating components AT THE SPECIFIED POSITION
top_components = []
for module_name, acts in component_acts.items():
    # acts shape: [1, seq_len, n_components]
    # Get activations only at the specified position
    acts_at_position = acts[0, token_position, :]  # [n_components]
    
    # Get top 10 components for this module at this position
    top_values, top_indices = torch.topk(acts_at_position, k=min(10, len(acts_at_position)))
    
    for idx, val in zip(top_indices, top_values):
        if val > 0:  # Only include components that actually activated
            top_components.append((module_name, idx.item(), val.item()))

# Sort by activation value
top_components.sort(key=lambda x: x[2], reverse=True)

target_modules = {
    # something w/ top-comopnent here
}

activations_storage, token_sequences = gather_component_activations_indexed(
    model=model,
    tokenizer=tokenizer,
    dataloader=dataloader,
    device=device,
    max_datapoints=1000,
    target_modules=target_modules
)

# Claude, Also get the original top-activations for the found module, components

# then maybe combine them? So show the icecream & then the top-activations for the found module, components

# Visualize each top component
print("\n\nVisualizations:")
for module_name, comp_idx, activation_value in top_components[:10]:  # Visualize top 10
    # Create storage format for visualization with just this component
    activations_storage_single = {
        module_name: component_acts[module_name][:, :, [comp_idx]].cpu()
    }
    token_sequences_single = input_ids.cpu()
    
    # Visualize
    html = visualize_top_component_activations(
        activations_storage=activations_storage_single,
        token_sequences=token_sequences_single,
        tokenizer=tokenizer,
        module_name=module_name,
        component_idx=0,  # 0 because we only have one component in the storage
        top_k=1,  # Just show this one example
        context_before=30,
        context_after=30
    )
    
    # Add the actual component index and activation value to the HTML title
    html = html.replace(
        f'<h2>Component 0 -', 
        f'<h2>Component {comp_idx} (activation at pos {token_position}: {activation_value:.3f}) -'
    )
    
    display(HTML(html))

In [ ]:
top_components

In [ ]:
# Your handwritten text
text = "Once upon a time, there was a little girl who loved to explore the forest."

# Tokenize
inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
input_ids = inputs['input_ids'].to(device)

# Get activations
with torch.no_grad():
    _, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
        input_ids,
        module_names=list(target_modules.keys())
    )

# Calculate component activations
As = {module_name: model.components[f"components.{module_name.replace('.', '-')}"].A 
      for module_name in target_modules.keys()}
component_acts = calc_component_acts(pre_weight_acts=pre_weight_acts, As=As)

# Convert to format expected by visualization function
# Create fake "storage" with just this one example
activations_storage_single = {}
for module_name, acts in component_acts.items():
    # acts shape: [1, seq_len, n_components]
    # For visualization, we need to select specific components
    if module_name in target_modules and target_modules[module_name] is not None:
        # Index into specific components
        acts = acts[:, :, target_modules[module_name]]
    activations_storage_single[module_name] = acts.cpu()

# Token sequences for this single example
token_sequences_single = input_ids.cpu()

# Find top components and visualize them
for module_name, acts in component_acts.items():
    acts = acts.squeeze(0)  # [seq_len, n_components]
    max_acts_per_component = acts.max(dim=0).values
    top_values, top_indices = torch.topk(max_acts_per_component, k=min(5, len(max_acts_per_component)))
    
    print(f"\nTop components for {module_name}:")
    for comp_idx, val in zip(top_indices, top_values):
        if val > 0:
            print(f"  Component {comp_idx.item()}: max activation {val.item():.3f}")
            
            # If this component is in our tracked list, visualize it
            if target_modules[module_name] is not None and comp_idx.item() in target_modules[module_name]:
                # Find its index in the tracked list
                tracked_idx = target_modules[module_name].index(comp_idx.item())
                
                html = visualize_top_component_activations(
                    activations_storage=activations_storage_single,
                    token_sequences=token_sequences_single,
                    tokenizer=tokenizer,
                    module_name=module_name,
                    component_idx=tracked_idx,
                    top_k=1,  # Just show this one example
                    context_before=15,
                    context_after=5
                )
                display(HTML(html))

In [ ]:
# Your handwritten text
text = "Once upon a time, there was a little girl who loved to explore the forest."

# Tokenize
inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64)
input_ids = inputs['input_ids'].to(device)

# Get activations
with torch.no_grad():
    _, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
        input_ids,
        module_names=list(target_modules.keys())
    )

# Calculate component activations
As = {module_name: model.components[f"{module_name.replace('.', '-')}"].A 
      for module_name in target_modules.keys()}
component_acts = calc_component_acts(pre_weight_acts=pre_weight_acts, As=As)

# Find top activating components
top_components = []
for module_name, acts in component_acts.items():
    # acts shape: [1, seq_len, n_components]
    acts = acts.squeeze(0)  # [seq_len, n_components]
    
    # Get max activation per component
    max_acts_per_component = acts.max(dim=0).values  # [n_components]
    
    # Get top 10 components for this module
    top_values, top_indices = torch.topk(max_acts_per_component, k=min(10, len(max_acts_per_component)))
    
    for idx, val in zip(top_indices, top_values):
        if val > 0:  # Only include components that actually activated
            top_components.append((module_name, idx.item(), val.item()))

# Sort by activation value
top_components.sort(key=lambda x: x[2], reverse=True)

# Print results
print(f"Text: {text}\n")
print("Top activating components:")
for module, comp_idx, activation in top_components[:20]:  # Show top 20
    print(f"{module} - Component {comp_idx}: {activation:.3f}")

In [ ]:
def gather_component_activations_indexed(
    model: ComponentModel,
    tokenizer: AutoTokenizer,
    dataloader: DataLoader,
    device: str = "cuda",
    max_datapoints: int = 10000,
    activation_threshold: float = 0.0,
) -> dict[str, dict[int, list[tuple[int, int, float]]]]:
    """
    Gather component activations indexed by component for efficient lookup.
    
    Returns:
        Dict mapping module_name -> component_idx -> list of (datapoint_idx, token_pos, activation_value)
        Only stores activations above threshold.
    """
    from tqdm import tqdm
    from collections import defaultdict
    
    # Get module names and components
    module_comp = [key.replace('-', '.') for key in model.components.keys()]
    components = {
        k.removeprefix("components.").replace("-", "."): v for k, v in model.components.items()
    }
    As = {module_name: components[module_name].A for module_name in components}
    
    # Storage: module_name -> component_idx -> list of (datapoint_idx, token_pos, activation)
    activations_by_component = {
        module_name: defaultdict(list) for module_name in module_comp
    }
    
    # Also store the actual token sequences for later retrieval
    token_sequences = []
    
    datapoint_idx = 0
    total_processed = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader):
            if total_processed >= max_datapoints:
                break
                
            input_ids = batch['input_ids'].to(device)
            batch_size, seq_len = input_ids.shape
            
            # Get activations
            _, pre_weight_acts = model.forward_with_pre_forward_cache_hooks(
                input_ids,
                module_names=module_comp
            )
            
            # Calculate component activations
            target_component_acts = calc_component_acts(
                pre_weight_acts=pre_weight_acts, 
                As=As
            )
            
            # Process each sequence in the batch
            for seq_idx in range(batch_size):
                # Store token sequence
                token_sequences.append(input_ids[seq_idx].cpu().tolist())
                
                # For each module
                for module_name, acts in target_component_acts.items():
                    # Get activations for this sequence: [seq_len, n_components]
                    seq_acts = acts[seq_idx]
                    
                    # Find all activations above threshold
                    positions, components = torch.where(seq_acts > activation_threshold)
                    
                    # Store each activation
                    for pos, comp_idx in zip(positions, components):
                        activation_value = seq_acts[pos, comp_idx].item()
                        activations_by_component[module_name][comp_idx.item()].append(
                            (datapoint_idx, pos.item(), activation_value)
                        )
                
                datapoint_idx += 1
                total_processed += 1
                
                if total_processed >= max_datapoints:
                    break
            
            # Clear GPU memory
            del input_ids, pre_weight_acts, target_component_acts
            torch.cuda.empty_cache()
    
    return activations_by_component, token_sequences

In [ ]:
model.model.transformer.h.7.

In [ ]:
(sparsity_masks['transformer.h.7.mlp.c_fc'] > 0.001).sum()

In [88]:
old_modules = {}
for component_name, component in components.items():
    module_name = component_name.replace("-", ".")
    # component: LinearComponent = self.components[module_name.replace(".", "-")]
    old_module = model.model.get_submodule(module_name)
    assert old_module is not None
    old_modules[module_name] = old_module

    # if masks is not None:
    component.mask = masks[component_name]
    model.model.set_submodule(module_name, component)

out = model(tokens.to(device))

# # Restore the original modules
for module_name, old_module in old_modules.items():
    model.model.set_submodule(module_name, old_module)

# Remove the masks attribute from the components
for component in components.values():
    component.mask = None

sec_out = model(tokens.to(device))

In [ ]:
(masks['transformer.h.7.mlp.c_fc'][0] > 0.0001).sum(dim=-1), (masks['transformer.h.7.mlp.c_proj'][0] > 0.01).sum(dim=-1)

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Literal
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import base64
from io import BytesIO
from IPython.display import HTML, display
import pandas as pd
from datasets import load_dataset

class ComponentActivationAnalyzer:
    def __init__(self, model, tokenizer, config, device='cuda'):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        
        # Extract components and gates
        self.module_comp = [key.replace('-', '.') for key in model.components.keys()]
        self.gates = {
            k.removeprefix("gates.").replace("-", "."): v 
            for k, v in model.gates.items()
        }
        self.components = {
            k.removeprefix("components.").replace("-", "."): v 
            for k, v in model.components.items()
        }
        self.As = {module_name: self.components[module_name].A for module_name in self.components}
        
        # Storage for activations
        self.activation_data = defaultdict(list)
        
    def collect_activations(self, dataset_split: str, num_samples: int = 1000, batch_size: int = 16):
        """Collect activation data from the dataset with batching."""
        # Load dataset
        dataset = load_dataset("roneneldan/TinyStories", split=dataset_split)
        
        print(f"Collecting activations from {num_samples} samples with batch size {batch_size}...")
        
        num_batches = (min(num_samples, len(dataset)) + batch_size - 1) // batch_size
        
        for batch_idx in tqdm(range(num_batches)):
            batch_start = batch_idx * batch_size
            batch_end = min((batch_idx + 1) * batch_size, num_samples, len(dataset))
            
            # Prepare batch
            batch_texts = []
            batch_tokens = []
            batch_indices = []
            
            for idx in range(batch_start, batch_end):
                text = dataset[idx]['text']
                tokens = self.tokenizer.encode(text, return_tensors='pt', 
                                              max_length=self.config.task_config.max_seq_len, 
                                              truncation=True)
                
                if tokens.shape[1] > 0:
                    batch_texts.append(text)
                    batch_tokens.append(tokens.squeeze(0))
                    batch_indices.append(idx)
            
            if not batch_tokens:
                continue
            
            # Pad sequences to same length
            max_len = max(t.shape[0] for t in batch_tokens)
            padded_tokens = []
            attention_masks = []
            
            for tokens in batch_tokens:
                pad_len = max_len - tokens.shape[0]
                if pad_len > 0:
                    padded = torch.cat([tokens, torch.full((pad_len,), self.tokenizer.pad_token_id)])
                    mask = torch.cat([torch.ones(tokens.shape[0]), torch.zeros(pad_len)])
                else:
                    padded = tokens
                    mask = torch.ones(tokens.shape[0])
                
                padded_tokens.append(padded)
                attention_masks.append(mask)
            
            # Stack into batch tensors
            batch_tensor = torch.stack(padded_tokens).to(self.device)
            attention_mask = torch.stack(attention_masks).to(self.device)
            
            with torch.no_grad():
                # Get activations for batch
                _, pre_weight_acts = self.model.forward_with_pre_forward_cache_hooks(
                    batch_tensor, module_names=self.module_comp
                )
                target_component_acts = calc_component_acts(pre_weight_acts=pre_weight_acts, As=self.As)
                
                # Get masks
                masks, sparsity_masks = calc_masks(
                    gates=self.gates, 
                    target_component_acts=target_component_acts, 
                    detach_inputs=False
                )
                
                # Process each item in batch
                for batch_item_idx in range(len(batch_texts)):
                    text = batch_texts[batch_item_idx]
                    dataset_idx = batch_indices[batch_item_idx]
                    tokens = batch_tokens[batch_item_idx].cpu().tolist()
                    valid_len = int(attention_mask[batch_item_idx].sum().item())
                    
                    # Store activation data for each component
                    for comp_name, mask_values in sparsity_masks.items():
                        # mask_values shape: [batch, seq_len, n_components]
                        item_mask = mask_values[batch_item_idx, :valid_len, :]
                        
                        # Find positions where activation > 0
                        activated_positions = (item_mask > 0).nonzero(as_tuple=True)
                        
                        if len(activated_positions[0]) > 0:
                            for pos_idx in range(len(activated_positions[0])):
                                seq_pos = activated_positions[0][pos_idx].item()
                                comp_idx = activated_positions[1][pos_idx].item()
                                activation_value = item_mask[seq_pos, comp_idx].item()
                                
                                self.activation_data[comp_name].append({
                                    'text': text,
                                    'tokens': tokens[:valid_len],  # Only store valid tokens
                                    'position': seq_pos,
                                    'component_idx': comp_idx,
                                    'activation': activation_value,
                                    'dataset_idx': dataset_idx
                                })
    
    def get_component_activations(self, 
                                component_name: str, 
                                component_idx: int,
                                sampling: Literal['top_k', 'stratified'] = 'top_k',
                                k: int = 10) -> List[Dict]:
        """Get activations for a specific component."""
        
        # Filter activations for this component
        comp_acts = [
            act for act in self.activation_data[component_name] 
            if act['component_idx'] == component_idx
        ]
        
        if not comp_acts:
            return []
        
        # Sort by activation value
        comp_acts.sort(key=lambda x: x['activation'], reverse=True)
        
        if sampling == 'top_k':
            return comp_acts[:k]
        
        elif sampling == 'stratified':
            # Divide into strata and sample
            n_strata = min(3, len(comp_acts))  # High, medium, low
            strata_size = len(comp_acts) // n_strata
            samples_per_stratum = k // n_strata
            remainder = k % n_strata
            
            sampled = []
            for i in range(n_strata):
                start_idx = i * strata_size
                end_idx = (i + 1) * strata_size if i < n_strata - 1 else len(comp_acts)
                stratum = comp_acts[start_idx:end_idx]
                
                n_samples = samples_per_stratum + (1 if i < remainder else 0)
                n_samples = min(n_samples, len(stratum))
                
                # Sample uniformly within stratum
                indices = np.linspace(0, len(stratum)-1, n_samples, dtype=int)
                sampled.extend([stratum[idx] for idx in indices])
            
            return sampled
    
    def visualize_activations(self,
                            component_name: str,
                            component_idx: int,
                            sampling: Literal['top_k', 'stratified'] = 'top_k',
                            k: int = 10,
                            context_before: int = 10,
                            context_after: int = 3,
                            save_png: bool = True,
                            filename: Optional[str] = None) -> HTML:
        """Create HTML visualization of component activations."""
        
        activations = self.get_component_activations(component_name, component_idx, sampling, k)
        
        if not activations:
            return HTML("<p>No activations found for this component.</p>")
        
        # Create HTML
        html_parts = [f"""
        <div style="font-family: Arial, sans-serif; padding: 20px; background-color: #f5f5f5;">
            <h2>Component: {component_name} [idx: {component_idx}]</h2>
            <p>Sampling method: {sampling}, showing {len(activations)} examples</p>
            <hr>
        """]
        
        for i, act in enumerate(activations):
            tokens = act['tokens']
            position = act['position']
            activation_value = act['activation']
            
            # Decode tokens with context
            start_pos = max(0, position - context_before)
            end_pos = min(len(tokens), position + context_after + 1)
            
            # Create colored HTML for tokens
            token_html = []
            for j in range(start_pos, end_pos):
                token_text = self.tokenizer.decode([tokens[j]])
                
                if j == position:
                    # Color based on activation strength (0 to 1)
                    blue_intensity = int(255 * (1 - activation_value))
                    color = f"rgb(0, 0, {255})"
                    bg_color = f"rgb({blue_intensity}, {blue_intensity}, 255)"
                    token_html.append(
                        f'<span style="background-color: {bg_color}; color: white; '
                        f'padding: 2px 4px; border-radius: 3px; font-weight: bold;">'
                        f'{token_text}</span>'
                    )
                else:
                    token_html.append(f'<span>{token_text}</span>')
            
            html_parts.append(f"""
            <div style="margin: 20px 0; padding: 15px; background-color: white; 
                        border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <p style="margin: 5px 0; color: #666;">
                    Example {i+1} | Activation: {activation_value:.4f} | 
                    Dataset idx: {act['dataset_idx']} | Token pos: {position}
                </p>
                <div style="font-size: 16px; line-height: 1.6; margin-top: 10px;">
                    {''.join(token_html)}
                </div>
            </div>
            """)
        
        html_parts.append("</div>")
        full_html = ''.join(html_parts)
        
        # Save as PNG if requested
        if save_png:
            if filename is None:
                filename = f"{component_name.replace('.', '_')}_{component_idx}_{sampling}.png"
            
            # Note: Saving HTML as PNG requires additional libraries like selenium
            # For now, we'll save the HTML file which can be opened and screenshot manually
            html_filename = filename.replace('.png', '.html')
            with open(html_filename, 'w', encoding='utf-8') as f:
                f.write(full_html)
            print(f"HTML saved to {html_filename} (screenshot manually or use selenium for PNG)")
        
        return HTML(full_html)
    
    def get_activation_summary(self) -> pd.DataFrame:
        """Get summary statistics of all component activations."""
        summary_data = []
        
        for comp_name, acts in self.activation_data.items():
            if not acts:
                continue
                
            # Group by component index
            comp_indices = defaultdict(list)
            for act in acts:
                comp_indices[act['component_idx']].append(act['activation'])
            
            for comp_idx, values in comp_indices.items():
                summary_data.append({
                    'component': comp_name,
                    'component_idx': comp_idx,
                    'num_activations': len(values),
                    'mean_activation': np.mean(values),
                    'max_activation': np.max(values),
                    'std_activation': np.std(values)
                })
        
        return pd.DataFrame(summary_data).sort_values('num_activations', ascending=False)


# Example usage function
def analyze_component_activations(model, tokenizer, config, device='cuda', batch_size=16):
    """Main function to run the analysis."""
    
    # Initialize analyzer
    analyzer = ComponentActivationAnalyzer(model, tokenizer, config, device)
    
    # Collect activations from dataset with batching
    # analyzer.collect_activations("train[:1000]", num_samples=1000, batch_size=batch_size)
    analyzer.collect_activations("train[:1000]", num_samples=10, batch_size=batch_size)
    
    # Get summary
    summary = analyzer.get_activation_summary()
    print("\nActivation Summary:")
    print(summary.head(20))
    
    # Example: Visualize a specific component
    # You can modify these parameters
    component_name = "transformer.h.3.mlp.c_proj"  # Example
    component_idx = 0  # Example component index
    
    # Top-k visualization
    html_top_k = analyzer.visualize_activations(
        component_name=component_name,
        component_idx=component_idx,
        sampling='top_k',
        k=10,
        context_before=10,
        context_after=3,
        save_png=True
    )
    display(html_top_k)
    
    # Stratified visualization
    html_stratified = analyzer.visualize_activations(
        component_name=component_name,
        component_idx=component_idx,
        sampling='stratified',
        k=9,  # Will give 3 high, 3 medium, 3 low
        context_before=10,
        context_after=3,
        save_png=True,
        filename=f"{component_name.replace('.', '_')}_{component_idx}_stratified.png"
    )
    display(html_stratified)
    
    return analyzer


# Run the analysis
    # Assuming your model, tokenizer, and config are already loaded


In [ ]:
analyzer = analyze_component_activations(model, tokenizer, config, device, batch_size=16)

In [ ]:
sparsity_masks['transformer.h.0.mlp.c_fc'].max(), (masks['transformer.h.0.mlp.c_fc'][0, -3] > 0).sum()

In [ ]:
vals = masks['transformer.h.0.mlp.c_fc'][0, 0]
# vals = target_component_acts['transformer.h.0.mlp.c_fc'][0, -1]
from matplotlib import pyplot as plt
plt.hist(vals.cpu().numpy(), bins = 100)
plt.show()

In [ ]:
vals.shape

In [ ]:
from spd.spd_types import WANDB_PATH_PREFIX, ModelPath
from pydantic import BaseModel
import yaml
from pathlib import Path
from spd.configs import Config

model_path = "out/all_kl_06-27_15.58_20250627_155830_844/model_50000.pth"

path = model_path
class ComponentModelPaths(BaseModel):
    """Paths to output files from a ComponentModel training run."""

    model: Path
    config: Path

paths = ComponentModelPaths(
    model=Path(path), config=Path(path).parent / "final_config.yaml"
)
out_dir = Path(path).parent

model_weights = torch.load(paths.model, map_location="cpu", weights_only=True)
with open(paths.config) as f:
    config = Config(**yaml.safe_load(f))
config.pretrained_model_class

In [ ]:
import torch
import yaml
from pathlib import Path
from spd.configs import Config
from spd.models.component_model import ComponentModel

def load_component_model_minimal(
    model_path: str,
    base_model,  # Your already-loaded base model
    target_module_patterns: list[str],
    m: int,
    n_gate_hidden_neurons: int | None = None,
    pretrained_model_output_attr: str | None = None
):
    """Load just the component model weights without reloading the base model."""
    
    # Load the saved weights
    model_weights = torch.load(model_path, map_location="cpu", weights_only=True)
    
    # Create the ComponentModel wrapper
    comp_model = ComponentModel(
        base_model=base_model,
        target_module_patterns=target_module_patterns,
        m=m,
        n_gate_hidden_neurons=n_gate_hidden_neurons,
        pretrained_model_output_attr=pretrained_model_output_attr,
    )
    
    # Load the component weights
    comp_model.load_state_dict(model_weights)
    
    return comp_model

# Usage example:
# Assuming you have your base model already loaded
# base_model = ... (your LLaMA or other model)
# comp_model = load_component_model_minimal(
#     model_path="out/all_kl_06-27_15.58_20250627_155830_844/model_50000.pth",
#     base_model=base_model,
#     target_module_patterns=["*.mlp.*", "*.self_attn.*"],  # adjust as needed
#     m=32,  # adjust as needed
# )